# 01 · Raw data acquisition

**Single responsibility:** download and verify official COCO 2017 instance annotations without downloading the full image archive

Run after the preceding numbered notebook unless the inputs already exist. Every generated artifact is written outside the notebook so this stage is reproducible.


In [1]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs/base.yaml').exists())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from vww_esp32.config import load_config, resolve_paths, seed_everything

config, ROOT = load_config(ROOT / 'configs/base.yaml')
paths = resolve_paths(config, ROOT)
seed_everything(config['project']['seed'])
ROOT


PosixPath('/Volumes/VM_SSD/Machine Learning/visual-wake-word-esp32')

## Source and license

This pipeline uses real MS COCO 2017 photographs. Review [COCO terms of use](https://cocodataset.org/#termsofuse); each image retains its source `license_id`. The annotation archive is about 241 MB. Images are selected and downloaded individually in notebook 03.

In [2]:
from vww_esp32.data import download_file, sha256_file

archive = paths['raw'] / 'annotations_trainval2017.zip'
archive = download_file(config['data']['annotation_url'], archive)
{'path': str(archive), 'bytes': archive.stat().st_size, 'sha256': sha256_file(archive)}

/Volumes/VM_SSD/Machine Learning/visual-wake-word-esp32/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
annotations_trainval2017.zip: 100%|██████████| 253M/253M [08:55<00:00, 473kB/s]  


{'path': '/Volumes/VM_SSD/Machine Learning/visual-wake-word-esp32/data/raw/coco2017/annotations_trainval2017.zip',
 'bytes': 252907541,
 'sha256': '113a836d90195ee1f884e704da6304dfaaecff1f023f49b6ca93c4aaae470268'}

In [3]:
from vww_esp32.data import extract_zip_safely

annotation_dir = paths['raw'] / 'annotations'
expected = [annotation_dir / 'instances_train2017.json', annotation_dir / 'instances_val2017.json']
if not all(path.exists() for path in expected):
    extract_zip_safely(archive, paths['raw'])
assert all(path.exists() and path.stat().st_size > 0 for path in expected)
expected

[PosixPath('/Volumes/VM_SSD/Machine Learning/visual-wake-word-esp32/data/raw/coco2017/annotations/instances_train2017.json'),
 PosixPath('/Volumes/VM_SSD/Machine Learning/visual-wake-word-esp32/data/raw/coco2017/annotations/instances_val2017.json')]

In [4]:
import json

raw_audit = {}
for annotation_path in expected:
    with annotation_path.open() as handle:
        payload = json.load(handle)
    raw_audit[annotation_path.stem] = {
        'images': len(payload['images']),
        'annotations': len(payload['annotations']),
        'categories': len(payload['categories']),
        'person_category_id': next(c['id'] for c in payload['categories'] if c['name'] == 'person'),
    }
raw_audit

{'instances_train2017': {'images': 118287,
  'annotations': 860001,
  'categories': 80,
  'person_category_id': 1},
 'instances_val2017': {'images': 5000,
  'annotations': 36781,
  'categories': 80,
  'person_category_id': 1}}